In [1]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
import hashlib

# --- PATHS ---
CSV_PATH  = r"C:\Users\Ramin\source\repos\Research Repo\ModeChoiceHybrid\Fastweb\Data\organized\4_2024_optionsMapped.csv"
OUT_HTML  = r"C:\Users\Ramin\source\repos\Research Repo\ModeChoiceHybrid\Fastweb\Data\Charts\locations_map.html"

# --- COLUMNS ---
COMPANY_ID  = "company_id"   # offices colored by this
OFFICE_ID   = "office_id"    # deduplicate offices by this + show in tooltip
EMPLOYEE_ID = "employee"     # show in tooltip for homes

LON_OFFICE  = "lon_office"
LAT_OFFICE  = "lat_office"
LON_HOME    = "lon_home"
LAT_HOME    = "lat_home"

# ---------- stable color per company_id (avoid red) ----------
OFFICE_COLOR_POOL = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
    "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173",
    "#3182bd", "#31a354", "#756bb1", "#636363"
]
def color_for_company(company_id) -> str:
    s = str(company_id).encode("utf-8")
    h = hashlib.md5(s).hexdigest()
    idx = int(h[:8], 16) % len(OFFICE_COLOR_POOL)
    return OFFICE_COLOR_POOL[idx]

# ---------- load ----------
df = pd.read_csv(CSV_PATH, encoding="ISO-8859-1")

# numeric coords
for c in [LON_OFFICE, LAT_OFFICE, LON_HOME, LAT_HOME]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

valid_office = df[LON_OFFICE].between(-180, 180) & df[LAT_OFFICE].between(-90, 90)
valid_home   = df[LON_HOME].between(-180, 180) & df[LAT_HOME].between(-90, 90)

df_home = df.loc[valid_home].copy()

# IMPORTANT: one point per office_id (take first occurrence)
df_office = (
    df.loc[valid_office]
      .dropna(subset=[OFFICE_ID])
      .drop_duplicates(subset=[OFFICE_ID], keep="first")
      .copy()
)

print(f"Valid homes:          {len(df_home)}/{len(df)}")
print(f"Unique valid offices: {len(df_office)} (dedup by {OFFICE_ID})")

# ---------- map center ----------
if len(df_office) > 0:
    center_lat = df_office[LAT_OFFICE].median()
    center_lon = df_office[LON_OFFICE].median()
elif len(df_home) > 0:
    center_lat = df_home[LAT_HOME].median()
    center_lon = df_home[LON_HOME].median()
else:
    raise ValueError("No valid office or home coordinates found.")

m = folium.Map(location=[center_lat, center_lon], zoom_start=6, tiles="CartoDB positron")

# Layers
homes_fg = folium.FeatureGroup(name="Homes (red)", show=True)
offices_fg = folium.FeatureGroup(name="Offices (unique by office_id)", show=True)

homes_cluster = MarkerCluster(name="Homes cluster").add_to(homes_fg)
offices_cluster = MarkerCluster(name="Offices cluster").add_to(offices_fg)

# ---------- homes: tooltip shows Employee ----------
has_emp = EMPLOYEE_ID in df.columns
for _, row in df_home.iterrows():
    emp_val = row.get(EMPLOYEE_ID) if has_emp else None
    tip = f"Employee: {emp_val}" if emp_val is not None else "Employee: (missing)"

    folium.CircleMarker(
        location=[row[LAT_HOME], row[LON_HOME]],
        radius=3,
        color="red",
        fill=True,
        fill_color="red",
        fill_opacity=0.45,
        tooltip=folium.Tooltip(tip, sticky=True)
    ).add_to(homes_cluster)

# ---------- offices (deduped): tooltip shows office_id only ----------
for _, row in df_office.iterrows():
    oid = row.get(OFFICE_ID)
    cid = row.get(COMPANY_ID)

    tip = f"office_id: {oid}"  # only this, as requested

    folium.CircleMarker(
        location=[row[LAT_OFFICE], row[LON_OFFICE]],
        radius=7,
        color="#000000",
        weight=1,
        fill=True,
        fill_color=color_for_company(cid),
        fill_opacity=0.9,
        tooltip=folium.Tooltip(tip, sticky=True)
    ).add_to(offices_cluster)

homes_fg.add_to(m)
offices_fg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

m.save(OUT_HTML)
print(f"✅ Saved interactive map -> {OUT_HTML}")


Valid homes:          1527/1527
Unique valid offices: 25 (dedup by office_id)
✅ Saved interactive map -> C:\Users\Ramin\source\repos\Research Repo\ModeChoiceHybrid\Fastweb\Data\Charts\locations_map.html
